In [23]:
%pip install mcp python-dotenv
%pip install langchain-ollama
# Ensure both servers are installed
%pip install mcp_weather_server

In [1]:
# Test to see if the local llm is running and reachable.
from langchain_ollama import ChatOllama
local_model="qwen3:0.6b"
local_llm = ChatOllama(
    model = local_model, 
    # Dont' make things up.
    temperature = 0,
    stream = True,
)
#print(local_llm.invoke("How is the weather in Tehran right now?"))
# Make sure the model name matches exactly what you see in `ollama list`
for chunk in local_llm.stream("How is the weather in Tehran right now?"):
    print(chunk.content, end="", flush=True)

In [2]:
# Test to see if the local llm is running and reachable.
from langchain_ollama import ChatOllama
fast_model="smollm2:135m"
fast_local_llm = ChatOllama(
    model = fast_model, 
    # Dont' make things up.
    temperature = 0,
    stream = True,
)

# Make sure the model name matches exactly what you see in `ollama list`
for chunk in fast_local_llm.stream("How is the weather in Tehran right now?"):
    print(chunk.content, end="", flush=True)

In [3]:
# Now lets start the weather mcp server.

import asyncio
import ollama
import json
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# 1. Define your MCP Server
server_params = StdioServerParameters(
    command="npx", 
    args=["-y", "@newsmcp/server"], 
    env=None
)

async def run_ollama_mcp():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            mcp_tools = await session.list_tools()
            ollama_tools = [
                {
                    "type": "function",
                    "function": {
                        "name": t.name,
                        "description": t.description,
                        "parameters": t.inputSchema,
                    },
                }
                for t in mcp_tools.tools
            ]
            
            messages = [
                {'role': 'system',
                 'content' : '''
                 You are a news curator. 
                  CRITICAL RULE: You MUST set 'per_page' to 10 when calling 'get_news'.
                 
                 Instructions:
                    1. Call `get_news` first.
                    2. Find the UUID for the top article and call `get_news_detail` immediately.
                    3. Provide a final summary based on the details.
                 '''},
                {"role": "user", "content": "What's the latest news from the USA?"}]
            
            while True:
                response = ollama.chat(
                    model=local_model, 
                    messages=messages,
                    tools=ollama_tools,
                )
                
                if not response.get('message', {}).get('tool_calls'):
                    messages.append(response['message'])
                    break

                messages.append(response['message'])
                
                for tool_call in response['message']['tool_calls']:
                    print(f"--- Executing: {tool_call['function']['name']} ---")
                    
                    result = await session.call_tool(
                        tool_call['function']['name'], 
                        tool_call['function']['arguments']
                    )
                    
                    # --- PRE-PROCESSING STEP ---
                    # Manually extract text to help the tiny model
                    raw_text = " ".join([c.text for c in result.content if hasattr(c, 'text')])
                    processed_text = raw_text
                    try:
                        data = json.loads(raw_text)
                        if isinstance(data, list):
                            processed_text = "\n".join([f"Title: {i.get('title')}\nSummary: {i.get('description')}" for i in data if isinstance(i, dict)])
                        elif isinstance(data, dict) and 'articles' in data:
                            processed_text = "\n".join([f"Title: {i.get('title')}\nSummary: {i.get('description')}" for i in data['articles']])
                    except: pass
                    
                    raw_len = len(processed_text)
                    print(f"--- Curating with {fast_model} ({raw_len} chars) ---")
                    
                    # --- CURATION STEP (Temperature 0) ---
                    curation_prompt = f"Extract ONLY the news headings and 1-sentence summaries from the following text. Remove all URLs and IDs. Be extremely concise:\n\n{processed_text}"
                    
                    curated_response = ollama.chat(
                        model=fast_model,
                        messages=[{'role': 'user', 'content': curation_prompt}],
                        options={'temperature': 0}
                    )
                    curated_content = curated_response['message']['content']
                    curated_len = len(curated_content)
                    
                    print(f"--- Reduced {raw_len} -> {curated_len} chars (Saved {raw_len - curated_len} chars) ---")
                    
                    messages.append({
                        'role': 'tool',
                        'content': curated_content,
                        'tool_call_id': tool_call.get('id') or tool_call.get('function', {}).get('id', 'default_id')
                    })
            
            print("\n--- FINAL CLEAN SUMMARY ---")
            print(messages[-1]['content'])

# Run in Jupyter
await run_ollama_mcp()


In [ ]:


import asyncio
import ollama
import json
import sys
from contextlib import AsyncExitStack
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# 1. Define all MCP Servers
servers_config = {
    "news": StdioServerParameters(
        command="npx", 
        args=["-y", "@newsmcp/server"], 
        env=None
    ),
    "weather": StdioServerParameters(
        command=sys.executable, 
        args=["-m", "mcp_weather_server"], 
        env=None
    )
}

async def run_multi_mcp_trip_planner():
    async with AsyncExitStack() as stack:
        sessions = {}
        all_ollama_tools = []
        tool_to_session = {} # Map tool name -> session
        
        # Initialize all servers
        for name, params in servers_config.items():
            print(f"--- Initializing {name} server ---")
            transport_read, transport_write = await stack.enter_async_context(stdio_client(params))
            session = await stack.enter_async_context(ClientSession(transport_read, transport_write))
            await session.initialize()
            sessions[name] = session
            
            # Aggregate tools
            mcp_tools = await session.list_tools()
            for t in mcp_tools.tools:
                tool_to_session[t.name] = session
                all_ollama_tools.append({
                    "type": "function",
                    "function": {
                        "name": t.name,
                        "description": t.description,
                        "parameters": t.inputSchema,
                    },
                })

        messages = [
            {'role': 'system', 'content': 'You are a travel planner. Use news and weather tools to provide a comprehensive trip plan. Be specific about locations like Tehran, Isfahan, or Shiraz.'},
            {'role': 'user', 'content': "Plan a trip to Iran, while doing so check the local conditions like weather and local news"}
        ]

        while True:
            response = ollama.chat(
                model=local_model, 
                messages=messages,
                tools=all_ollama_tools,
            )
            
            if not response.get('message', {}).get('tool_calls'):
                messages.append(response['message'])
                break

            messages.append(response['message'])
            
            for tool_call in response['message']['tool_calls']:
                tool_name = tool_call['function']['name']
                print(f"--- Executing: {tool_name} ---")
                
                # Route to correct session
                session = tool_to_session.get(tool_name)
                if not session:
                    result_content = f"Error: Tool {tool_name} not found."
                else:
                    result = await session.call_tool(tool_name, tool_call['function']['arguments'])
                    result_content = " ".join([c.text for c in result.content if hasattr(c, 'text')])
                
                # Add tool result to conversation
                messages.append({
                    'role': 'tool',
                    'content': result_content,
                    'tool_call_id': tool_call.get('id') or tool_call.get('function', {}).get('id', 'default_id')
                })
        
        print("\n--- TRIP PLAN ---")
        print(messages[-1]['content'])

# Execute the planner
await run_multi_mcp_trip_planner()



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
--- Initializing news server ---
--- Initializing weather server ---
--- Executing: get_current_weather ---
--- Executing: get_news ---
